# Projeto: Análise, filtragem e reconstrução de áudio musical com Fourier e Laplace

**Integrante:** Bruno de Cunto Rezende  
**Disciplina:** Série e Transformada de Fourier / Sistemas Lineares  
**Ambiente:** Google Colab (Python)

## Objetivo
Analisar um áudio musical real, identificar sua frequência fundamental e harmônicos por FFT, projetar e aplicar um filtro passa-baixas, reconstruir o sinal pela Transformada Inversa de Fourier e relacionar o filtro digital ao modelo analógico por Transformada de Laplace.

> **Arquivo necessário:** um áudio próprio ou livre de direitos, preferencialmente uma nota musical sustentada (piano, violão etc.) em formato WAV. No Colab, a etapa 2 permite enviar o arquivo. Se o áudio estiver em MP3, converta-o para WAV no Audacity antes de enviar. Não use áudio protegido sem autorização para publicar o projeto.


## Sumário
1. Série de Fourier de uma onda quadrada e fenômeno de Gibbs.
2. FFT do áudio musical real e interpretação do espectro.
3. Filtro passa-baixas e reconstrução com IFFT.
4. Modelo contínuo \(H(s)\), polos, zeros e estabilidade.
5. Conclusões, limitações e checklist de entrega.

**Como executar:** abra o notebook no Google Colab e execute as células em ordem. A etapa 1 funciona sem arquivo. Na etapa 2, envie um arquivo WAV quando solicitado. Os resultados numéricos dependem do áudio escolhido e da frequência de corte configurada.


## Preparação — bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
from IPython.display import Audio, display
from google.colab import files
import os

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=4, suppress=True)

print("Bibliotecas carregadas.")


# Etapa 1 — Aproximando funções com Série de Fourier

Usaremos uma onda quadrada periódica de período \(2\pi\), definida por \(f(t)=1\) para \(0<t<\pi\) e \(f(t)=-1\) para \(-\pi<t<0\). Ela é ímpar, portanto seus coeficientes de cosseno e o termo constante são zero. Os coeficientes seno são \(b_n=4/(n\pi)\) para \(n\) ímpar e zero para \(n\) par.

Assim, a série trigonométrica é:
\[
f(t)\sim \frac{4}{\pi}\left(\sin t+\frac{\sin 3t}{3}+\frac{\sin 5t}{5}+\cdots\right).
\]
Na forma complexa, \(f(t)\sim\sum_{k=-\infty}^{\infty}c_k e^{ikt}\), com \(c_0=0\) e \(c_k=(1-(-1)^k)/(i\pi k)\) para \(k\ne0\). Isso equivale a \(c_k=2/(i\pi k)\) para \(k\) ímpar e zero para \(k\) par.

**Interpretação esperada:** mais harmônicos tornam a aproximação melhor nas regiões contínuas. Próximo às descontinuidades, aparecem oscilações e um sobressinal que não desaparece completamente ao aumentar o número de termos: é o fenômeno de Gibbs. Exatamente no salto, a série converge para a média dos limites laterais, que neste caso é zero.


In [ ]:
# Onda quadrada e aproximações com diferentes quantidades de harmônicos ímpares
t = np.linspace(-np.pi, np.pi, 3000, endpoint=False)
onda = signal.square(t)
N_valores = [1, 3, 5, 10, 25, 50]

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True, sharey=True)
for ax, N in zip(axes.flat, N_valores):
    aprox = np.zeros_like(t)
    for n in range(1, N + 1, 2):
        aprox += (4 / (np.pi * n)) * np.sin(n * t)
    ax.plot(t, onda, label="Onda quadrada", linewidth=1.5)
    ax.plot(t, aprox, label=f"Série: até n={N}", linewidth=1.2)
    ax.set_title(f"N máximo = {N}")
    ax.set_xlabel("t (rad)")
    ax.set_ylabel("Amplitude")
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Coeficientes trigonométricos b_n e coeficientes complexos c_k
N_coef = 15
n = np.arange(1, N_coef + 1)
b_n = np.where(n % 2 == 1, 4 / (np.pi * n), 0.0)

plt.figure()
plt.stem(n, b_n)
plt.title("Coeficientes seno b_n da onda quadrada")
plt.xlabel("n (harmônico)")
plt.ylabel("b_n")
plt.show()

k = np.arange(-15, 16)
c_k = np.zeros(k.shape, dtype=complex)
for i, ki in enumerate(k):
    if ki != 0:
        c_k[i] = (1 - (-1)**ki) / (1j * np.pi * ki)

plt.figure()
plt.stem(k, np.abs(c_k))
plt.title("Magnitude dos coeficientes complexos |c_k|")
plt.xlabel("k")
plt.ylabel("|c_k|")
plt.show()

print("Para a onda quadrada ímpar: a0 = 0, a_n = 0.")
print("b_n = 4/(n*pi) para n ímpar; b_n = 0 para n par.")
print("c_0 = 0; c_k = (1 - (-1)^k)/(i*pi*k), para k != 0.")


### Comentário dos gráficos — Etapa 1
Os gráficos mostram que a soma de mais harmônicos representa melhor as partes planas e as transições da onda quadrada. Perto dos saltos, a oscilação característica de Gibbs permanece, embora fique concentrada em uma região cada vez menor. Os coeficientes diminuem aproximadamente como \(1/n\), e os harmônicos pares são nulos devido à simetria da onda. Isso ilustra como funções seno/cosseno ortogonais podem servir de base para representar sinais periódicos.


# Etapa 2 — Do tempo para a frequência: FFT do áudio real

Envie um arquivo WAV. O código converte áudio estéreo para mono, normaliza a amplitude e exibe um trecho temporal e o espectro unilateral. A resolução aproximada entre bins é \(\Delta f=f_s/N\), e a frequência de Nyquist é \(f_N=f_s/2\). Componentes acima de Nyquist não podem ser representadas corretamente e podem sofrer aliasing.

**Atenção:** para identificar com clareza uma nota, prefira um trecho relativamente estável, sem fala ou muitos instrumentos simultâneos. O pico de maior amplitude pode ser a fundamental, mas em alguns timbres um harmônico pode ser mais intenso que ela.


In [ ]:
# Envie um arquivo WAV do computador.
uploaded = files.upload()
wav_files = [name for name in uploaded.keys() if name.lower().endswith(".wav")]
if not wav_files:
    raise ValueError("Nenhum arquivo .wav foi enviado. Converta o áudio para WAV e execute novamente esta célula.")

audio_path = wav_files[0]
fs, audio = wavfile.read(audio_path)
print(f"Arquivo: {audio_path}")
print(f"Taxa de amostragem: {fs} Hz")

# Converter inteiros para float e áudio estéreo para mono
if np.issubdtype(audio.dtype, np.integer):
    info = np.iinfo(audio.dtype)
    escala = max(abs(info.min), info.max)
    x = audio.astype(np.float64) / escala
else:
    x = audio.astype(np.float64)

if x.ndim == 2:
    x = x.mean(axis=1)

x = x - np.mean(x)  # remove componente DC
pico = np.max(np.abs(x))
if pico > 0:
    x = x / pico

duracao = len(x) / fs
nyquist = fs / 2
print(f"Duração: {duracao:.2f} s | Amostras: {len(x)}")
print(f"Frequência de Nyquist: {nyquist:.1f} Hz")

# Para tornar os gráficos e o processamento mais rápidos, analisar até 10 s.
max_segundos = 10
x_analise = x[:min(len(x), int(max_segundos * fs))]
tempo = np.arange(len(x_analise)) / fs

plt.figure()
plt.plot(tempo, x_analise, linewidth=0.7)
plt.title("Áudio musical no domínio do tempo")
plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude normalizada")
plt.show()

N = len(x_analise)
X = np.fft.rfft(x_analise)
freqs = np.fft.rfftfreq(N, d=1/fs)
magnitude = np.abs(X) / N
if len(magnitude) > 2:
    magnitude[1:-1] *= 2

plt.figure()
plt.plot(freqs, magnitude)
plt.title("Espectro de amplitude (FFT)")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Amplitude relativa")
plt.xlim(0, min(nyquist, 5000))
plt.show()

# Listar os 10 maiores picos, ignorando DC
picos, _ = signal.find_peaks(magnitude)
picos = picos[freqs[picos] > 20]
top = picos[np.argsort(magnitude[picos])[-10:]][::-1] if len(picos) else np.array([], dtype=int)
print("Principais picos espectrais:")
for idx in top:
    print(f"{freqs[idx]:8.2f} Hz  | amplitude relativa = {magnitude[idx]:.5f}")

display(Audio(x_analise, rate=fs))


### Como interpretar — Etapa 2
- **Picos estreitos** indicam componentes sinusoidais marcantes; em uma nota musical, podem corresponder à fundamental e aos harmônicos.
- **Picos aproximadamente múltiplos** de uma frequência sugerem uma série harmônica, embora o timbre e o ruído alterem suas amplitudes.
- **Energia espalhada** pode vir do ataque do instrumento, reverberação, ruído de gravação ou de vários instrumentos.
- A FFT descreve o conteúdo de frequência do trecho analisado. Um trecho curto melhora a localização temporal, mas reduz a resolução em frequência; um trecho longo melhora a resolução, desde que o sinal permaneça suficientemente estável.
- Não conclua que todo componente alto é ruído: os harmônicos fazem parte do timbre musical.


# Etapa 3 — Filtro passa-baixas e reconstrução pela IFFT

Aplicaremos um filtro Butterworth passa-baixas de segunda ordem. Ele preserva as frequências abaixo da frequência de corte e atenua as mais altas de maneira gradual. Ajuste `fc` de acordo com o espectro observado: a frequência de corte deve ser escolhida com base no objetivo, pois um corte muito baixo pode remover harmônicos importantes e deixar o áudio abafado.

Para demonstrar explicitamente a Transformada Inversa de Fourier, calculamos a resposta em frequência do filtro digital, multiplicamos a FFT do sinal pela resposta e usamos `irfft` para reconstruir o sinal filtrado. Como o filtro tem fase, essa implementação em frequência equivale à aplicação da resposta complexa do sistema ao trecho considerado.


In [ ]:
# Configure a frequência de corte após observar o espectro.
# Valor inicial de exemplo: 1500 Hz. Deve ser menor que Nyquist.
fc = min(1500.0, 0.4 * fs)
ordem = 2
if not (0 < fc < fs/2):
    raise ValueError("A frequência de corte deve estar entre 0 e Nyquist.")

sos = signal.butter(ordem, fc, btype="lowpass", fs=fs, output="sos")

# Resposta complexa do filtro digital exatamente nas frequências dos bins da rFFT
omega = 2 * np.pi * freqs / fs
_, H_bins = signal.sosfreqz(sos, worN=omega)

X_filtrado = X * H_bins
y = np.fft.irfft(X_filtrado, n=N)

# Resposta em frequência para gráfico
f_resp, H_resp = signal.sosfreqz(sos, worN=4096, fs=fs)
plt.figure()
plt.plot(f_resp, 20*np.log10(np.maximum(np.abs(H_resp), 1e-8)))
plt.axvline(fc, linestyle="--", label=f"fc = {fc:.0f} Hz")
plt.title("Resposta em frequência do filtro passa-baixas")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Magnitude (dB)")
plt.ylim(-80, 5)
plt.legend()
plt.show()

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax[0].plot(tempo, x_analise, linewidth=0.7)
ax[0].set_title("Antes do filtro")
ax[0].set_ylabel("Amplitude")
ax[1].plot(tempo, y, linewidth=0.7)
ax[1].set_title("Depois do filtro (IFFT)")
ax[1].set_xlabel("Tempo (s)")
ax[1].set_ylabel("Amplitude")
plt.tight_layout()
plt.show()

Y = np.fft.rfft(y)
mag_y = np.abs(Y) / N
if len(mag_y) > 2:
    mag_y[1:-1] *= 2

plt.figure()
plt.plot(freqs, magnitude, label="Original", alpha=0.8)
plt.plot(freqs, mag_y, label="Filtrado", alpha=0.8)
plt.title("Comparação dos espectros antes e depois")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Amplitude relativa")
plt.xlim(0, min(nyquist, 5000))
plt.legend()
plt.show()

# Métricas simples
rms_original = np.sqrt(np.mean(x_analise**2))
rms_filtrado = np.sqrt(np.mean(y**2))
erro_rms = np.sqrt(np.mean((x_analise-y)**2))
print(f"Frequência de corte: {fc:.1f} Hz")
print(f"RMS original: {rms_original:.4f}")
print(f"RMS filtrado: {rms_filtrado:.4f}")
print(f"RMS da diferença (original - filtrado): {erro_rms:.4f}")

print("Áudio original:")
display(Audio(x_analise, rate=fs))
print("Áudio filtrado:")
display(Audio(y, rate=fs))

# Exportar trecho filtrado como WAV para download
from scipy.io.wavfile import write
y_export = np.int16(np.clip(y / max(np.max(np.abs(y)), 1e-12), -1, 1) * 32767)
wav_saida = "audio_filtrado.wav"
write(wav_saida, fs, y_export)
print(f"Arquivo gerado: {wav_saida}")


### Comentário dos resultados — Etapa 3
A resposta em frequência deve ficar próxima de 0 dB na banda passante e cair acima da frequência de corte. No espectro, as componentes altas devem diminuir; no tempo, a forma de onda pode ficar mais suave. Compare os dois áudios para avaliar se o filtro reduziu componentes indesejadas sem eliminar demasiados harmônicos relevantes.

O filtro passa-baixas não distingue automaticamente “ruído” de “música”: ele atenua qualquer componente acima do corte, incluindo harmônicos úteis. Portanto, a escolha de `fc` é um compromisso entre redução de componentes altas e preservação do timbre. A IFFT reconstrói o sinal correspondente ao espectro filtrado.


# Etapa 4 — Sistema linear, Transformada de Laplace e polos/zeros

Um filtro passa-baixas Butterworth analógico de segunda ordem pode ser descrito por:
\[
H(s)=\frac{\omega_c^2}{s^2+\sqrt{2}\,\omega_c s+\omega_c^2},
\qquad \omega_c=2\pi f_c.
\]
O numerador constante implica que não há zeros finitos. Os polos são as raízes do denominador:
\[
s_{1,2}=-\frac{\omega_c}{\sqrt{2}}\pm j\frac{\omega_c}{\sqrt{2}}.
\]
Ambos têm parte real negativa, então o sistema analógico causal é BIBO estável. A resposta em magnitude cai quando a frequência aumenta; a posição dos polos no semiplano esquerdo determina a dinâmica e contribui para essa resposta.

**Importante:** o processamento de áudio da etapa 3 usa um filtro digital Butterworth projetado com a taxa de amostragem. A função \(H(s)\) abaixo é o modelo analógico Butterworth de referência, com a mesma ordem e frequência de corte. É uma comparação conceitual consistente, mas não é literalmente o mesmo sistema discreto: a transformação bilinear relaciona os modelos e pode introduzir prewarping da frequência de corte.


In [ ]:
from scipy import signal
import sympy as sp

fc_analog = fc
wc = 2 * np.pi * fc_analog
s = sp.symbols('s', complex=True)
H_s = wc**2 / (s**2 + np.sqrt(2)*wc*s + wc**2)

print("Função de transferência analógica H(s):")
display(sp.factor(H_s))

# Polos e zeros analógicos
num = [wc**2]
den = [1, np.sqrt(2)*wc, wc**2]
zeros = np.roots(num) if len(num) > 1 else np.array([])
polos = np.roots(den)
print("Zeros finitos:", zeros if len(zeros) else "Nenhum")
print("Polos:", polos)
print("Partes reais dos polos:", np.real(polos))
print("Estável?", bool(np.all(np.real(polos) < 0)))

plt.figure()
if len(zeros):
    plt.scatter(np.real(zeros), np.imag(zeros), marker='o', facecolors='none', edgecolors='C0', s=100, label='Zeros')
plt.scatter(np.real(polos), np.imag(polos), marker='x', s=100, label='Polos')
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("Mapa de polos e zeros — filtro analógico")
plt.xlabel("Parte real")
plt.ylabel("Parte imaginária")
plt.legend()
plt.axis('equal')
plt.show()

# Resposta analógica em frequência H(jw)
w = np.logspace(np.log10(max(1, wc/100)), np.log10(wc*100), 2000)
_, H_analog = signal.freqs(num, den, worN=w)
plt.figure()
plt.semilogx(w/(2*np.pi), 20*np.log10(np.maximum(np.abs(H_analog), 1e-10)))
plt.axvline(fc_analog, linestyle='--', label=f"fc = {fc_analog:.1f} Hz")
plt.axhline(-3.0103, linestyle=':', label="-3 dB")
plt.title("Resposta em frequência do modelo analógico")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Magnitude (dB)")
plt.ylim(-80, 5)
plt.legend()
plt.show()

# Expressão simbólica simplificada em termos de wc
wc_sym = sp.symbols('omega_c', positive=True, real=True)
H_symbolic = wc_sym**2 / (s**2 + sp.sqrt(2)*wc_sym*s + wc_sym**2)
print("Expressão simbólica:")
display(H_symbolic)


### Interpretação — Etapa 4
O mapa mostra um par de polos conjugados no semiplano esquerdo e nenhum zero finito. Como a parte real dos polos é negativa, as respostas naturais decaem com o tempo e o sistema analógico é estável. A frequência de corte define a escala dos polos: aumentar \(f_c\) afasta os polos da origem e desloca a região de transição para frequências maiores. A queda da magnitude acima de \(f_c\) é coerente com a função passa-baixas observada na etapa 3.

A função de transferência contínua relaciona a Transformada de Laplace da saída e da entrada, sob condições iniciais nulas: \(H(s)=Y(s)/X(s)\). A resposta em frequência é obtida avaliando \(s=j\omega\), isto é, \(H(j\omega)\).


# Conclusão

Neste projeto, a Série de Fourier mostrou como uma soma de funções ortogonais representa uma onda periódica e como o fenômeno de Gibbs aparece nas descontinuidades. A FFT permitiu observar as componentes espectrais de um áudio musical real. Em seguida, o filtro passa-baixas atenuou as frequências acima do corte e a IFFT reconstruiu o sinal filtrado. Por fim, a função de transferência analógica de segunda ordem e seu mapa de polos e zeros relacionaram a resposta em frequência à estabilidade do sistema.

**Personalize esta conclusão antes da entrega:** cite a frequência fundamental aproximada encontrada, a frequência de corte escolhida, quais componentes diminuíram e o que você percebeu ao ouvir o áudio filtrado. Não afirme que houve remoção de ruído sem verificar os gráficos e ouvir os resultados.
